# 1. Preparación, entrenamiento y trazas
Ejecutar en Linux con el kernel de `.venv`. Los datos y modelos quedan locales e ignorados por Git. Este notebook llama a los adaptadores del protocolo; los notebooks históricos son solo referencia.


In [ ]:
from pathlib import Path
import platform
import sys
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
assert Path.cwd().resolve() == ROOT
print(platform.platform(), sys.version.split()[0], ROOT)
assert sys.platform.startswith('linux'), 'Entrenar LSTM en el entorno Linux validado'


## Datos físicos congelados
Si faltan los archivos locales, revisar primero su tamaño y procedencia en `data_contract/manifests/imported_files.json`; la hidratación verifica hashes y no modifica la fuente.


In [ ]:
# Ejecutar solo si faltan los archivos ignorados y existe el checkout fuente local.
# from shared.data_preparation.hydrate import hydrate
# hydrate(Path('../Clasificaci-n-y-correcci-n'))
contract = validate_splits()
assert [contract['splits'][s]['rows'] for s in ('train', 'validation', 'test')] == [172873, 28800, 31680]
print(contract['data_source'], contract['manifest_id'])


## LightGBM con noche y contexto interior/exterior
Es un experimento nuevo de 68 variables. El v8 original permanece como referencia de 66 variables.


In [ ]:
from approaches.lightgbm.night_context import train as train_lightgbm, export_test as export_lightgbm
lightgbm_run = train_lightgbm()
lightgbm_trace, lightgbm_isolated = export_lightgbm(lightgbm_run)
print(lightgbm_run, lightgbm_trace, lightgbm_isolated, sep='\n')


## LSTM causal
Model 1 y Model 2 se entrenan con train; validation selecciona checkpoints y umbral. Test solo se usa al exportar trazas. Puede tardar bastante.


In [ ]:
from approaches.lstm.adapter import train as train_lstm, export_test as export_lstm
lstm_run = train_lstm()
lstm_trace, lstm_isolated = export_lstm(lstm_run)
print(lstm_run, lstm_trace, lstm_isolated, sep='\n')


Guardar las rutas de ambos runs. El notebook 02 verifica la paridad de objetivos y genera el informe comparativo. No versionar las salidas de este notebook.
